# Plotting Parrots

## Initialising codebase

In [21]:
import sys
import os
from datetime import datetime as dt
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import plotly.express as px

### Import data

In [22]:

beaufort_scale = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/beaufort_scale.csv')
birds = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/birds.csv', low_memory=False)
sea_states = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/sea_states.csv')
ships = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/ships.csv')

### Check if data imported correctly

### Beaufort_scale

In [23]:
beaufort_scale.head()

,wind_speed_class,wind_description,wind_speed_knots_min,wind_speed_knots_max
0,0,calm,0,1.0
1,1,light air,1,3.0
2,2,light breeze,4,6.0
3,3,gentle breeze,7,10.0
4,4,moderate breeze,11,16.0


### Birds

In [24]:
birds.head()

,bird_observation_id,record_id,species_common_name,species_scientific_name,species_abbreviation,age,wan_plumage_phase,plumage_phase,sex,count,...,sitting_on_ship,in_hand,n_flying_past,flying_past,n_accompanying,accompanying,n_following_ship,following_ship,moulting,naturally_feeding
0,1,1083001,Royal / Wandering albatross,Diomedea epomophora / sanfordi / antipodensis ...,DIOEPOSANANTEXU,NaN,NaN,NaN,NaN,6.0,...,False,False,0.0,False,6.0,True,0.0,False,NaN,False
1,2,1083001,Black-browed albatross sensu lato,Diomedea impavida / melanophrys,DIOIMPMEL,NaN,NaN,NaN,NaN,2.0,...,False,False,0.0,False,2.0,True,0.0,False,NaN,False
2,3,1083001,Cape petrel,Daption capense,DAPCAP,NaN,NaN,NaN,NaN,8.0,...,False,False,0.0,False,8.0,True,0.0,False,NaN,False
3,4,1083001,Fairy prion,Pachyptila turtur,PACTUR,NaN,NaN,NaN,NaN,2.0,...,False,False,0.0,False,2.0,True,0.0,False,NaN,False
4,5,1083001,Sooty shearwater,Puffinus griseus,PUFGRI,NaN,NaN,NaN,NaN,4.0,...,False,False,0.0,False,4.0,True,0.0,False,NaN,False


In [25]:
sea_states.head()

,sea_state_class,sea_state_description,wave_meters_min,wave_meters_max
0,0,"calm, glassy",0.0,0.0
1,1,"calm, rippled",0.0,0.1
2,2,smooth,0.1,0.5
3,3,slight,0.5,1.2
4,4,moderate,1.3,2.5


In [26]:
ships.head()

,record_id,date,time,latitude,longitude,hemisphere,activity,speed,direction,cloud_cover,...,wind_speed_class,wind_direction,air_temperature,pressure,sea_state_class,sea_surface_temperature,depth,observer,census_method,season
0,1083001,1975-10-15,14:00:00,-45.917,165.400,E,"steaming, sailing",15.0,NaN,overcast,...,5.0,NaN,NaN,NaN,5.0,NaN,NaN,D. Jeffcock,full,spring
1,1084001,1975-11-03,13:10:00,-35.533,125.000,E,"steaming, sailing",14.0,NaN,partially cloudy,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
2,1084002,1975-11-04,14:20:00,-37.667,132.250,E,"steaming, sailing",14.5,NaN,overcast,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
3,1084003,1975-11-08,16:15:00,-40.000,162.000,E,"steaming, sailing",14.6,NaN,partially cloudy,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
4,1086001,1975-11-16,12:30:00,-36.167,174.917,E,"steaming, sailing",15.0,NaN,partially cloudy,...,0.0,NaN,NaN,NaN,1.0,NaN,NaN,D. Jeffcock,full,spring


The analysis uses 4 datasets: 
1. birds.csv, containing individual seabird observation records; 
2. ships.csv, containing information about ship-based censuses, including geographical coordinates, dates and seasons; 
3. beaufort_scale.csv, containing description of the wind condition corresponsing to the integers in the beaufort scale class in ships.csv; 
4. sea_states.csv, containing description of the sea states corresponding to the integers in the sea state class in ships.csv.

### Question: How is the distibution of seabirds affected by the factors: weather conditions, season and geographical location

# Data Cleaning

## Genus names
Genus is the first word of `species_scientific_name`, standardised to the [IOC World Bird List v15.2](https://www.worldbirdnames.org/) (fixes misspellings, renamed genera and species that moved genus).

* Rows with no scientific name are unidentified birds, so they are dropped.
* Names that span two genera (e.g. "Procellaria parkinsoni / Puffinus carneipes") are dropped.
* "Puffinus sp." (Shearwater, unidentified) and "Sterna sp." (Tern, unidentified) are dropped: after the IOC split, an unidentified shearwater could be *Ardenna*, *Puffinus* or *Calonectris*, and an unidentified tern could be one of several genera.
* Rows with no `count` are **kept**: the bird was seen, only the number was not written down. They still count as a sighting, and sums or medians of `count` skip them.
* A `count` of 99999 means "estimated over 100,000" in the dataset's data dictionary, so these are real, very large flocks and are kept.

Individual seabird species were classified into their respective genera to facilitate broader comparisons across the 3 factors and reduce the complexity of the dataset.

In [27]:
birds_cleaned = birds.copy()

birds_cleaned["Genus"] = birds_cleaned["species_scientific_name"].str.split().str[0]

In [28]:
# Misspelt genus
GENUS_TYPOS = {
    "Thallassarche": "Thalassarche",
    "Phaeton":       "Phaethon",
    "Magadyptes":    "Megadyptes",
}

# Genus renamed in the IOC World Bird List
IOC_GENUS_RENAMES = {
    "Catharacta":   "Stercorarius",
    "Stictocarbo":  "Phalacrocorax",
    "Procelsterna": "Anous",
    "Lugensa":      "Aphrodroma",
}

# Changed genera by IOC
IOC_SPECIES_MOVES = {
    "Diomedea impavida":     "Thalassarche",
    "Diomedea melanophrys":  "Thalassarche",
    "Puffinus griseus":      "Ardenna",
    "Puffinus tenuirostris": "Ardenna",
    "Puffinus carneipes":    "Ardenna",
    "Puffinus pacificus":    "Ardenna",
    "Puffinus bulleri":      "Ardenna",
    "Larus novaehollandiae": "Chroicocephalus",
    "Larus bulleri":         "Chroicocephalus",
    "Sterna bergii":         "Thalasseus",
    "Sterna albifrons":      "Sternula",
    "Sterna nereis":         "Sternula",
}

def ioc_genera(name): 
    genera, genus = set(), ""
    for part in name.split("/"):
        words = part.split()
        if not words:
            continue
        # epithet is the word after genus which is required for changing the genus
        if words[0][0].isupper():   # part names its own genus: "Genus species"
            genus, epithet = words[0], (words[1] if len(words) > 1 else "") # epithet is the name after the genus
        else:                       # "Genus a / b": b shares the previous genus
            epithet = words[0]
        fixed = GENUS_TYPOS.get(genus, IOC_GENUS_RENAMES.get(genus, genus))
        genera.add(IOC_SPECIES_MOVES.get(f"{genus} {epithet}", fixed))
    return sorted(genera)   # unique genera, as a list

n_start = len(birds_cleaned)
print(f"Starting rows: {n_start} | Genera: {birds_cleaned['Genus'].nunique()}\n")

# Drop unidentified bird species
before = len(birds_cleaned)
birds_cleaned = birds_cleaned.dropna(subset=["species_scientific_name"]).copy()
print(f"1. No scientific name dropped: {before - len(birds_cleaned)} rows ({len(birds_cleaned)} remaining)\n")

# Drop names that span two genera
genera = birds_cleaned["species_scientific_name"].apply(ioc_genera)
multiple_genera = genera.apply(len) > 1
print(f"2. Spanning two genera dropped: {multiple_genera.sum()} rows ({(~multiple_genera).sum()} remaining)")
print(birds_cleaned.loc[multiple_genera, "species_scientific_name"].value_counts().to_string(), "\n")
birds_cleaned = birds_cleaned[~multiple_genera].copy()

# Assign IOC genus (typos and renames applied)
birds_cleaned["Genus_original"] = birds_cleaned["Genus"]
birds_cleaned["Genus"] = genera[~multiple_genera].str[0]
changed = birds_cleaned[birds_cleaned["Genus"] != birds_cleaned["Genus_original"]]
print(f"3. Genus changed to IOC: {len(changed)} rows | Genera: {birds_cleaned['Genus_original'].nunique()} -> {birds_cleaned['Genus'].nunique()}")
print(changed.groupby(["Genus_original", "Genus"]).size().rename("rows").to_string(), "\n")

# Drop unidentified shearwaters and terns: after the IOC split their genus is unknown
unknown_genus = birds_cleaned["species_scientific_name"].isin(["Puffinus sp.", "Sterna sp."])
print(f"4. Genus unknown after IOC split dropped: {unknown_genus.sum()} rows ({(~unknown_genus).sum()} remaining)")
print(birds_cleaned.loc[unknown_genus, "species_common_name"].value_counts().to_string(), "\n")
birds_cleaned = birds_cleaned[~unknown_genus].copy()

# Keep records without a count: the bird was seen but its number was not written down,
# so it is still a sighting. Sums and medians of "count" skip these rows automatically
print(f"5. No count kept as sightings: {birds_cleaned['count'].isna().sum()} rows\n")

# Note that count = 99999 means "estimated over 100,000" (data dictionary),
# so these rows are real, very large flocks and are kept

print(f"Total dropped: {n_start - len(birds_cleaned)} | Rows remaining: {len(birds_cleaned)} | Genera: {birds_cleaned['Genus'].nunique()}")

Starting rows: 49019 | Genera: 47

1. No scientific name dropped: 1101 rows (47918 remaining)

2. Spanning two genera dropped: 4 rows (47914 remaining)
species_scientific_name
Procellaria parkinsoni / Puffinus carneipes              2
Procellaria parkinsoni / Pterodroma macroptera gouldi    1
Eudyptes / Magadyptes sp                                 1 

3. Genus changed to IOC: 10611 rows | Genera: 47 -> 46
Genus_original  Genus          
Catharacta      Stercorarius        168
Diomedea        Thalassarche       3610
Larus           Chroicocephalus     788
Lugensa         Aphrodroma           10
Phaeton         Phaethon              5
Procelsterna    Anous                27
Puffinus        Ardenna            5135
Sterna          Sternula              3
                Thalasseus          402
Stictocarbo     Phalacrocorax        33
Thallassarche   Thalassarche        430 

4. Genus unknown after IOC split dropped: 130 rows (47784 remaining)
species_common_name
Shearwater (unidentified)  

## Ships
Keep full counts only. Partial counts are shorter or casual observations, so they are not comparable. We drop all records from ships.csv with census column "partial"

In [29]:
ships_cleaned = ships.copy()
print(f"Starting counts: {len(ships_cleaned)}\n")

# Keep full counts only 
partial = ships_cleaned["census_method"] != "full"
print(f"1. Partial counts dropped: {partial.sum()} ({(~partial).sum()} remaining)")
ships_cleaned = ships_cleaned[~partial].copy()

Starting counts: 12310

1. Partial counts dropped: 199 (12111 remaining)


## Match birds to the cleaned ships
Keep only bird records from the counts kept above.

In [30]:
# Keep birds from full counts only
before = len(birds_cleaned)
birds_cleaned = birds_cleaned[birds_cleaned["record_id"].isin(ships_cleaned["record_id"])].copy()
print(f"Bird rows from dropped counts removed: {before - len(birds_cleaned)} ({len(birds_cleaned)} remaining)")

Bird rows from dropped counts removed: 536 (47248 remaining)


## Top 5 genera
Genera are ranked by the **number of censuses they were seen in** (unique `record_id`), so a genus seen often ranks above one seen in a few huge flocks. This avoids giving disproportionate weight to exceptionally large flocks, particularly because the dataset contains observations for count capped at 99,999 birds. Total birds and the median flock size are shown for reference.

In [31]:
# Summarise each genus: how many censuses it was seen in, total birds, and typical flock size
genus_summary = (birds_cleaned.groupby("Genus")
                 .agg(n_census=("record_id", "nunique"),
                      total_birds=("count", "sum"),
                      median_flock=("count", "median"))
                 .sort_values("n_census", ascending=False)
                 .reset_index())

top5_genus = genus_summary.head(5)
top5_genus

,Genus,n_census,total_birds,median_flock
0,Diomedea,5806,20534.0,1.0
1,Thalassarche,4540,21754.0,1.0
2,Ardenna,4074,1242843.0,3.0
3,Pterodroma,3987,17388.0,2.0
4,Daption,2945,26026.0,4.0


# Plotting by geographical location

### Purpose
Does **where** a count was taken change which bird genera were seen? This plot uses only latitude and longitude. Season is the bar plot's variable, so the seasons here are averaged out rather than compared.

The five genera come from the shared **Top 5 genera** section above.

In [32]:
birds_by_geo_loc = birds_cleaned.merge(ships_cleaned[["record_id", "latitude", "longitude"]], on="record_id", how="left")
top5_birds_by_geo_loc = birds_by_geo_loc[birds_by_geo_loc["Genus"].isin(top5_genus["Genus"])]
top5_birds_by_geo_loc.head()

,bird_observation_id,record_id,species_common_name,species_scientific_name,species_abbreviation,age,wan_plumage_phase,plumage_phase,sex,count,...,n_accompanying,accompanying,n_following_ship,following_ship,moulting,naturally_feeding,Genus,Genus_original,latitude,longitude
0,1,1083001,Royal / Wandering albatross,Diomedea epomophora / sanfordi / antipodensis ...,DIOEPOSANANTEXU,NaN,NaN,NaN,NaN,6.0,...,6.0,True,0.0,False,NaN,False,Diomedea,Diomedea,-45.917,165.4
1,2,1083001,Black-browed albatross sensu lato,Diomedea impavida / melanophrys,DIOIMPMEL,NaN,NaN,NaN,NaN,2.0,...,2.0,True,0.0,False,NaN,False,Thalassarche,Diomedea,-45.917,165.4
2,3,1083001,Cape petrel,Daption capense,DAPCAP,NaN,NaN,NaN,NaN,8.0,...,8.0,True,0.0,False,NaN,False,Daption,Daption,-45.917,165.4
4,5,1083001,Sooty shearwater,Puffinus griseus,PUFGRI,NaN,NaN,NaN,NaN,4.0,...,4.0,True,0.0,False,NaN,False,Ardenna,Puffinus,-45.917,165.4
5,6,1084001,Royal albatross sensu lato,Diomedea epomophora / sanfordi,DIOEPOSAN,NaN,NaN,NaN,NaN,10.0,...,0.0,False,8.0,True,NaN,False,Diomedea,Diomedea,-35.533,125.0


### Step 2: one row per count, with its location and quarter
* Keep only **full** counts (12,111 of 12,310). The 199 partial counts are shorter, so they are not comparable.
* Drop the 11 full counts with no coordinates, leaving **12,100**.
* Tag each count with its quarter of the year: Jan–Mar (3,457 counts), Apr–Jun (2,123), Jul–Sep (2,506), Oct–Dec (4,014).
* For each count and group, record True if the group was logged at least once. A group counts once per count, irregardless of however many birds were seen.

In [33]:
# full censuses with coordinates, tagged with their quarter of the year
geo_loc = ships_cleaned[["record_id", "latitude", "longitude", "date"]].dropna()
geo_loc["quarter"] = pd.to_datetime(geo_loc["date"]).dt.quarter          # 1 = Jan–Mar ... 4 = Oct–Dec

In [34]:
# one row per census, True/False for each group
present = top5_birds_by_geo_loc.assign(present=True).pivot_table(index="record_id", columns="Genus", values="present",
                                                         aggfunc="any", fill_value=False)
cells = geo_loc.merge(present, left_on="record_id", right_index=True, how="left").fillna(False)

### Step 3: plot smoothed presence-rate maps with interactive hover
**Measure: presence rate**, the share of counts in a patch of sea that logged the group. We use a rate rather than raw sightings because ships surveyed some lanes far more often than others, so raw sightings would mostly map ship traffic.

1. **Grid.** Counts are binned into 0.1° cells (about 10 km).
2. **Smooth.** Both grids (all counts, and counts with the group) are blurred with a Gaussian filter (σ = 6 cells ≈ 0.6°), and the rate is one divided by the other. Each patch therefore draws on counts within roughly 60 km, which fills the gaps between neighbouring ship tracks without inventing data far from any ship.
3. **Weight the seasons equally.** The rate is worked out separately for each quarter, then the four are averaged. Without this, the Tasman Sea (39% of its counts in Jan–Mar but only 14% in Jul–Sep) would look like a summer place, and season would leak into a map that is meant to be about location.
4. **Only colour well-surveyed sea.** A cell is shown only if it had at least 6 counts nearby in **every** quarter. Blank sea means "not enough data", and not "no birds". About 27% of the map window fulfills the criteria, and the coloured areas follow the ship routes.
5. **Map window.** 147–180°E, 30–48.5°S keeps 86% of all full counts, including the Tasman crossings between Australia and New Zealand.
6. **Summary number.** The number in each map's corner is the median rate over all coloured sea, i.e. the rate at a typical surveyed spot.


In [35]:
from scipy.ndimage import gaussian_filter
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from shapely.geometry import box
np.seterr(divide="ignore", invalid="ignore")

LON, LAT = (147, 180), (-48.5, -30) # map window zoomed in to cover only NZ surroundings and tasman sea between NZ and AU
lon_bins, lat_bins = np.arange(145, 182, 0.1), np.arange(-50.5, -28, 0.1)   # 0.1° grid, 2° spare around the window

def smooth_count(df, weights=None):
    """Count censuses in each 0.1° cell, then blur over ~0.6° (6 cells) so gaps between ship tracks fill in."""
    counts, _, _ = np.histogram2d(df["latitude"], df["longitude"], bins=[lat_bins, lon_bins], weights=weights)
    return gaussian_filter(counts, sigma=6) * 226

quarters = [cells[cells["quarter"] == q] for q in [1, 2, 3, 4]]
effort = [smooth_count(df) for df in quarters]
surveyed = np.all([e >= 6 for e in effort], axis=0)  # only keep sea with >= 6 censuses nearby in every quarter

rate = {}
for g in top5_genus["Genus"]:
    # % of censuses that logged g, per quarter, then the average of the four quarters (each season has equal weightage)
    per_quarter = [smooth_count(df, df[g].astype(float)) / e * 100 for df, e in zip(quarters, effort)]
    rate[g] = np.where(surveyed, np.mean(per_quarter, axis=0), np.nan)

lon_mid, lat_mid = lon_bins[:-1] + 0.05, lat_bins[:-1] + 0.05
rows = np.where((lat_mid >= LAT[0]) & (lat_mid <= LAT[1]))[0][::2]
cols = np.where((lon_mid >= LON[0]) & (lon_mid <= LON[1]))[0][::2]
inside = np.ix_(rows, cols)
nearby = sum(effort)[inside].round().astype("float32")
order = ["Diomedea", "Thalassarche", "Ardenna", "Pterodroma", "Daption"]


In [36]:
# trace coastline
land = gpd.read_file("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_land.geojson")
land = land.clip(box(LON[0] - 3, LAT[0] - 3, LON[1] + 3, LAT[1] + 3)).explode().simplify(0.02)
coast_x, coast_y = [], []
for poly in land:
    x, y = poly.exterior.xy
    coast_x += list(x) + [None]; coast_y += list(y) + [None]

# red color variations
reds = [[0, "#fef0d9"], [0.25, "#fdcc8a"], [0.5, "#fc8d59"], [0.75, "#e34a33"], [1, "#b30000"]]

fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.02, vertical_spacing=0.1,
                    subplot_titles=[f"<b>{g}</b>" for g in order])
for k, g in enumerate(order):
    r, c = k // 3 + 1, k % 3 + 1
    fig.add_trace(go.Contour(x=lon_mid[cols], y=lat_mid[rows], z=rate[g][inside].round().astype("float32"), customdata=nearby,
                             zmin=0, zmax=100, hoverongaps=False, line_width=0,
                             contours=dict(start=0, end=100, size=20), autocontour=False,
                             colorscale=reds, showscale=(k == 0), name=g,
                             hovertemplate=f"<b>{g}</b><br>lat %{{y:.1f}}, lon %{{x:.1f}}<br>"
                                           "Recorded in %{z:.0f}% of counts here<br>Based on %{customdata:.0f} nearby counts<extra></extra>",
                             colorbar=dict(orientation="h", x=0.835, y=0.1, len=0.3, thickness=14, tickvals=[0, 20, 40, 60, 80, 100],
                                           ticktext=["0%", "20%", "40%", "60%", "80%", "100%"],
                                           title=dict(text="% of 10-minute counts<br>that recorded the group", side="top"))),
                  row=r, col=c)
    fig.add_trace(go.Scatter(x=coast_x, y=coast_y, fill="toself", fillcolor="white", mode="lines",
                             line=dict(color="#5b6470", width=0.7), hoverinfo="skip", showlegend=False), row=r, col=c)

    ax = f"{k + 1 if k else ''}"
    fig.add_annotation(x=LON[1] - 0.6, y=LAT[0] + 0.6, xref=f"x{ax}", yref=f"y{ax}", xanchor="right", yanchor="bottom",
                       align="right", showarrow=False, bgcolor="rgba(255,255,255,0.85)", borderpad=4,
                       text=f"<span style='font-size:20px;color:#b30000'><b>{np.nanmedian(rate[g]):.0f}%</b></span><br>"
                            "<span style='font-size:10px;color:dimgrey'>typical spot</span>")
    fig.update_xaxes(range=LON, visible=False, row=r, col=c)
    fig.update_yaxes(range=LAT, visible=False, scaleanchor=f"x{k + 1 if k else ''}", scaleratio=1.3, row=r, col=c)

fig.add_annotation(x=157, y=-45.5, text="<i>Tasman Sea</i>", showarrow=False, font=dict(color="grey"), xref="x", yref="y")
fig.add_annotation(x=0.69, y=0.40, xref="paper", yref="paper", xanchor="left", align="left", showarrow=False,
                   font=dict(size=12, color="dimgrey"),
                   text="<b>How to read</b><br>"
                        "Darker = the group was recorded more often there.<br>"
                        "Big number = the typical (median) value across<br>all coloured sea. Hover a map for exact numbers.<br><br>"
                        "<b>Note:</b><br>"
                        "Coloured areas follow ship routes.<br>Blank sea had too few counts in at least one season.")
fig.update_layout(width=1050, height=640, plot_bgcolor="#e9eef2", paper_bgcolor="white", margin=dict(l=10, r=10, t=95, b=10),
                  title=dict(text="<b>Albatrosses rule the open sea; shearwaters and gulls stay near the coast; petrels are everywhere</b>"
                                  "<br><span style='font-size:13px;color:dimgrey'>How often each bird group was recorded in 10-minute counts from ships, 1969–1990. "
                                  "All four seasons are count equally.</span>",
                             x=0.01, font=dict(size=17)))
fig.show()

### Step 4: near the coast vs open sea
To put numbers on the pattern in the map, each coloured grid cell is given its distance to the nearest coastline and placed in a band: < 50 km, 50–150 km or > 150 km. The table gives each group's median rate in each band. Distances are approximate (1° ≈ 100 km at these latitudes).

In [37]:
# Take the same rates, but split by distance from the coast
# For each surveyed grid cell, distance to land = distance from the cell's center to the nearest coastline
from shapely.geometry import box
coast = land.clip(box(140, -55, 185, -25)).union_all()
grid = gpd.GeoSeries(gpd.points_from_xy(*[a.ravel() for a in np.meshgrid(lon_mid, lat_mid)]))
km = grid.distance(coast).to_numpy().reshape(surveyed.shape) * 100   # 1° is about 100 km at these latitudes
band = pd.cut(km[surveyed], [0, 50, 150, np.inf], labels=["< 50 km", "50–150 km", "> 150 km"])
pd.DataFrame({g: rate[g][surveyed] for g in order}).groupby(band, observed=True).median().round(0).T

,< 50 km,50–150 km,> 150 km
Diomedea,22.0,37.0,80.0
Thalassarche,36.0,31.0,37.0
Ardenna,42.0,41.0,18.0
Pterodroma,20.0,27.0,44.0
Daption,18.0,18.0,25.0


### Checks and limitations
* **Sensitivity.** We re-ran the map with a narrower and a wider blur (σ = 4 or 8 cells) and a lower and a higher effort threshold (3 or 12 counts per quarter). Albatrosses and petrels stay the most widespread groups, and prions stay low everywhere. The medians for shearwaters and gulls & terns rise under the strictest settings, because those keep mostly near-shore sea, where these groups are common. That is the same coastal pattern as in Step 4.
* **Data collection is concentrated around common shipping routes, and are not a random sample.** Ships followed fixed lanes, so the map describes the sea along those routes, not the whole region.
* **Years differ by area.** 76% of Tasman counts are from 1975–1984, while 89% of counts in New Zealand waters are from 1980–1989, so part of a difference between areas could be a difference between decades, ships or observers.
* **Data comes from observations, and do not depict true abundance.** Large birds and birds that follow ships are easier to spot than small ones, so rates compare how often a group was *recorded*.
* **Location is not independent from weather.** Wind and sea state vary from place to place, which overlaps slightly with the weather plot.

# Helper functions

In [38]:
def top_5(dataframe, column):
    return None